Axioms for the theory of <a class="ProveItLink" href="theory.ipynb">proveit.physics.quantum.QEC2</a>
========

In [ ]:
import proveit
# Prepare this notebook for defining the axioms of a theory:
%axioms_notebook # Keep this at the top following 'import proveit'.

from proveit                 import (b, e, f, i, l, n, s, A, B, G, X,
                                     ExprRange, ExprTuple, IndexedVar)
from proveit.core_expr_types import A_1_to_n
from proveit.logic           import And, Equals, Exists, Forall, InSet
from proveit.logic.sets      import EmptySet, Intersect, Disjoint, SymmetricDifference, Union
from proveit.numbers         import (zero, one, two, three, Add, greater_eq,
                                     Interval, LessEq, Mod, Mult, Natural, Neg,
                                     subtract)
from proveit.graphs          import IsGraph, IsPath
from proveit.linear_algebra  import AntiCommutator

from proveit.physics.quantum.QEC2 import (
        _ell, _max_buf_weight, ActionFunction, BufiloSequences, BufiloSets,
        CheckFunction, EdgeFaults, Errors, f_i, f_one_to_n, f_one_to_n_minus_one,
        Faults, Realizations, s_i, s_i_plus_one, s_one_to_n, s_prime,
        State, StateAction, States, StateSyndrome, Weight)


In [ ]:
%begin axioms

In [ ]:
weight_in_natural = Forall(A, InSet(Weight(A), Natural))

In [ ]:
weight_empty_set = Equals(Weight(EmptySet), zero)

In [ ]:
weight_additivity = Forall(n,
       Forall(A_1_to_n,
              Equals(Weight(Union(A_1_to_n)),
                     Add(ExprRange(i, Weight(IndexedVar(A, i)), one, n))),
       conditions=[Disjoint(A_1_to_n)]),
domain=Natural)

In [ ]:
binary_weight_additivity = (
    Forall((A, B),
           Equals(Weight(Union(A, B)),
                          Add(Weight(A), Weight(B), Neg(Weight(Intersect(A, B)))))
    )
)

#### Axiomatic Definition of BUFILO (in terms of membership)

In [ ]:
bufs_membership_def = (
    Forall(b,
           Equals(InSet(b, BufiloSets),
                And(InSet(b, Errors),
                    Equals(CheckFunction(b), EmptySet),
                    Equals(ActionFunction(_ell, b), one))
           )
    )
)

#### Axiomatic Definition of Irreducible BUFILOs, $i$<span style="font-variant: small-caps;">bufs</span>

In [ ]:
from proveit import b
from proveit.logic import InSet, NotExists
from proveit.logic.sets import SetOfAll, SubsetProper
from proveit.physics.quantum.QEC2 import b_prime, BufiloSets, IrreducibleBufiloSets
irreducible_bufs_def = (
    Equals(IrreducibleBufiloSets, SetOfAll(b, b,
                                           conditions=[NotExists(b_prime, SubsetProper(b_prime, b),
                                           domain=BufiloSets)],
           domain=BufiloSets))
)

#### Axiomatic Definition of Irreducible BUFILOs (in terms of membership)

##### SEE Theorems notebook

#### Axiomatic Characterization of BUFILO weight limit $w_{\text{BUF}}$

In [ ]:
_max_buf_weight_in_natural = InSet(_max_buf_weight, Natural)

In [ ]:
_max_buf_weight_ge_three = greater_eq(_max_buf_weight, three)

#### Axiomatic Characterization of BUFILO Sequence, $f \in \mathcal{F}_{\ell, w_{\text{BUF}}}^{\text{seq}}$

In [ ]:
anti_commutation_buf_seq_first_elem = (
    Forall(n,
    Forall(f_one_to_n,
           Equals(AntiCommutator(IndexedVar(f, one), _ell),
                  zero),
    condition=InSet(ExprTuple(f_one_to_n), BufiloSequences), domain=Faults),
    conditions = [LessEq(n, _max_buf_weight)], domain = Natural)
)

#### Errors (Sets of Faults)

An error $e \in \text{ERRS}$ is simply a set of faults.

In [ ]:
from proveit import e
from proveit.logic.sets import Set
errors_membership_def = (
    Forall(e,
           Equals(InSet(e, Errors),
                  Exists(n,
                         Exists((f_one_to_n),
                                Equals(e, Set(f_one_to_n)),
                         domain=Faults),
                  domain=Natural))
    )
)

#### Augmented Syndrome States

In [ ]:
from proveit.physics.quantum.QEC2 import ErrorState
state_membership_def = Forall(s, Equals(InSet(s, States), Exists(e, Equals(s, ErrorState(_ell, e)), domain = Errors)))

In [ ]:
state_def = (
    Forall(e, Equals(ErrorState(_ell, e), ExprTuple(CheckFunction(e), ActionFunction(_ell,e))),
           domain=Errors
          )
)

#### EdgeFaults, EdgeFaultsMembership

In [ ]:
edge_faults_membership_def = (
    Forall((s, s_prime),
           Forall(f, 
                  Equals(InSet(f, EdgeFaults(s, s_prime)),
                         And(Equals(StateSyndrome(s_prime),
                                    SymmetricDifference(StateSyndrome(s), CheckFunction(Set(f)))),
                             Equals(StateAction(s_prime),
                                    Mod(Add(StateAction(s), ActionFunction(_ell, Set(f))), two)))).with_wrap_after_operator()
           ),
    domain=States)
)

#### `all_states_graph`

Notice that the all-states graph $G_{S_{\ell}}$ is designed to generate or represent “irreducible BUFILOs,” or BUFILOs that do not properly contain _other_ BUFILOs, due to the second condition: once the graph reaches state $(\emptyset, 1)$, the choice function $\nu$ fails to have any further detectors to eliminate.

In [ ]:
from proveit import s, X, Y, Function
from proveit.logic import And, Or
from proveit.logic.sets import Difference, Set, SetOfAll
from proveit.graphs import Graph
from proveit.physics.quantum.QEC2 import (
        _nu, ActionFunction, s_prime, StateAction, States, StateSyndrome)
# Define the all_states_graph expression to use further below
all_states_graph = Graph(States, SetOfAll((s, s_prime), ExprTuple(s, s_prime),
         conditions=[Or(
             And(Equals(StateAction(s), zero), Equals(StateAction(s_prime), one)),
             And(Equals(StateAction(s), one),
                 InSet(Function(_nu, s), Difference(StateSyndrome(s), StateSyndrome(s_prime))))
         ).with_wrap_after_operator()],
         domain=States))

In [ ]:
from proveit.physics.quantum.QEC2 import AllStatesGraph
all_states_graph_def = Equals(AllStatesGraph, all_states_graph)

#### Realizations & RealizationsMembership

In [ ]:
realizations_membership_def = (
    Forall(G,
       Forall(n,
              Forall((s_one_to_n),
                     Forall(f_one_to_n_minus_one,
                            Equals(InSet(ExprTuple(f_one_to_n_minus_one),
                                         Realizations(ExprTuple(s_one_to_n), G)),
                                   And(ExprRange(i, InSet(f_i, EdgeFaults(s_i, s_i_plus_one)),
                                                 one, subtract(n, one)))).with_wrap_after_operator(),
                     domain=Faults).with_wrapping(),
              conditions=[IsPath(ExprTuple(s_one_to_n), G)]).with_wrapping(),
       domain = Natural).with_wrapping(),
    conditions=[IsGraph(G)]).with_wrapping()
)

In [ ]:
realizations_def = (
    Forall(G,
    Forall(n,
    Forall((s_one_to_n),
            Equals(Realizations(ExprTuple(s_one_to_n), G),
                   SetOfAll(f_one_to_n_minus_one, (f_one_to_n_minus_one),
                            conditions=[Forall(i, InSet(f_i, EdgeFaults(s_i, s_i_plus_one)),
                                        domain=Interval(one, subtract(n, one)))],
                   domain=Faults)).with_wrap_after_operator(),
    conditions=[IsPath((s_one_to_n), G)]).with_wrapping(),
    domain=Natural).with_wrapping(),
    conditions=[IsGraph(G)]).with_wrapping()
)

In [ ]:
%end axioms